In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0, "d_model, h'ye tam bölünmeli"
        self.h = h
        self.d_k = d_model // h
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # tüm kafaların W_Q'su yan yana
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # çıkış projeksiyonu

    def forward(self, x_q, x_kv, mask=None):
        # x_q: (B, n_q, d_model)   x_kv: (B, n_k, d_model)
        B, n_q, _ = x_q.shape
        n_k = x_kv.shape[1]
        # 1) Projeksiyon sonra kafalara böl
        Q = self.W_Q(x_q).view(B, n_q, self.h, self.d_k).transpose(1, 2)   # (B, h, n_q, d_k)
        K = self.W_K(x_kv).view(B, n_k, self.h, self.d_k).transpose(1, 2)  # (B, h, n_k, d_k)
        V = self.W_V(x_kv).view(B, n_k, self.h, self.d_k).transpose(1, 2)  # (B, h, n_k, d_k)
        # 2) Her kafada ölçekli dot-product attention
        scores = Q @ K.transpose(-2, -1) / self.d_k ** 0.5                 # (B, h, n_q, n_k)
        if mask is not None:
            scores = scores + mask                                         # (n_q, n_k) yayınlanır
        weights = F.softmax(scores, dim=-1)                                # (B, h, n_q, n_k)
        heads = weights @ V                                                # (B, h, n_q, d_k)
        # 3) Concat: kafaları yan yana birleştir
        concat = heads.transpose(1, 2).contiguous().view(B, n_q, -1)       # (B, n_q, h*d_k = d_model)
        # 4) W_O
        return self.W_O(concat), weights                                   # (B, n_q, d_model)

torch.manual_seed(0)
mha = MultiHeadAttention(d_model=512, h=8)
x = torch.randn(2, 10, 512)                  # batch=2, n=10 token
out, w = mha(x, x)
print("çıktı:", out.shape)
print("ağırlıklar:", w.shape)
print("parametre sayısı:", sum(p.numel() for p in mha.parameters()))

çıktı: torch.Size([2, 10, 512])
ağırlıklar: torch.Size([2, 8, 10, 10])
parametre sayısı: 1048576
